<a href="https://colab.research.google.com/github/Matheus-Santos-AI/FarmTech-na-Era-da-Cloud-Computing/blob/main/Cap_1_FarmTech_na_Era_da_Cloud_Computing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Integrantes:
- Matheus de S. Santos Rm566901
- Ricardo José Amorin Rm567312
- Klaus Lohany Barbosa de Oliveira
- Victor Oliveira Fedeli Tate Rm566823
- Paulo Roberto Silva Amaral Ribeiro Junior Rm568413

BIBLIOTECAS

In [ ]:
# Instalações
!pip install pycaret
!pip install lime
!pip install shap
# Apenas para evitar warnings
import warnings
warnings.filterwarnings('ignore')
from pycaret.regression import*
# Bibliotecas de Data Science
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
# Separação dos dados
from sklearn.model_selection import train_test_split
# Modelos a serem utilizados
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
# Avaliação de modelos
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from lime import lime_tabular
import shap

In [ ]:
#importação de dados do arquivo csv
data = pd.read_csv('crop_yield.csv')
data_predict = data

##ANALISE DOS DADOS

In [ ]:
#   - Informações gerais do dataset
display(data.info())
#   - Análises estatísticas gerais
display(data.describe())
#   - Identificação de duplicatas
display(f"Número de linhas duplicadas: {data.duplicated().sum()}")

Podemos constatar que o dataframe não possui dados nulos ou linhas duplicadas, se tratando de um dataset bem 'comportado'

In [ ]:
#VERIFICANDO OUTLIERS
#   - Selecionar as variáveis para o boxplot
features = ['Precipitation (mm day-1)',	'Specific Humidity at 2 Meters (g/kg)', 'Relative Humidity at 2 Meters (%)',	'Temperature at 2 Meters (C)',	'Yield']
#   - Criar uma figura com 2 linhas e 4 colunas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
#   - Iterar pelas variáveis e criar o boxplot em cada subplot
for i, feature in enumerate(features):
    row = i // 3
    col = i % 3
    sns.boxplot(y=data[feature], ax=axes[row, col])
    axes[row, col].set_title(feature)
#   - Ajustar o layout da figura
plt.tight_layout()
plt.show()

Apesar de possuir outliers na feature 'Yield' não iremos tratar pois é importante a precisam dos numeros desse caso e possuimos poucas linhas no dataset

In [ ]:
#CORRELAÇÃO ENTRE AS FEATURES
# Plot da matriz de correlação com mapa de calor
plt.figure(figsize=(12, 10))
sns.heatmap(data.corr(numeric_only=True),
            annot=True,
            cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação')
plt.show()

In [ ]:
#Removendo features que não tem grande impacto
data.drop(columns=['Relative Humidity at 2 Meters (%)'], inplace=True)

Como as features 'Specific Humidity at 2 Meters (g/kg)' e 'Relative Humidity at 2 Meters (%)'são diretamente relacionadas podemos escolher entre uma delas e desconsiderar a outra tendo em vista que não representa impacto no modelo e dessa forma tambem redusimos a quantidade total de dados para ele processar

In [ ]:
# Plotar o desbalanceamento da variável "Crop"
plt.figure(figsize=(6, 4))
sns.countplot(x='Crop', data=data)
plt.title('Distribuição da variável Crop')
plt.xlabel('Crop')
plt.ylabel('Contagem')
plt.show()

avaliando a quantidade de cada cultivo (Crop), avaliamos se exite desbalanceamento consideravel entre eles , como os graficos anteriores podem mostrar , todos tem a mesma quantidade de dados

##CLASSIFICAÇÃO(CLUSTERIZAÇÃO) ATRAVÈS DE VARIOS MODELOS DIFERENTES

In [ ]:

# Inicializando o encoder
le = LabelEncoder()

# Aplicando e criando uma nova coluna
data['crop_encoded'] = le.fit_transform(data['Crop'])



Aqui transformamos a feature 'Crop' em numeral atraves do encoded para melhor desempenho do modelo

In [ ]:
data = data.drop(columns=['Crop'])

In [ ]:
data

In [ ]:
# Separando os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(data.drop('crop_encoded', axis= 1),
                 data['crop_encoded'],
                 test_size=0.2,
                 random_state=42)
# Padronizando as features
ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)

Foi separado os dados em treino e teste sendo 80% de treino e 20% de teste, após isso foi padronizado atraves do StandardScaler para melhor adequação dos modelos

In [ ]:
#FUNÇÕES PARA AVALIAR OS MODELOS
# Avaliando os modelos de classificação, calculando algumas métricas
def avaliar_modelo(y_true, y_pred, average):
    # Acurácia NÃO recebe o parâmetro average
    accuracy = accuracy_score(y_true, y_pred)

    # As outras métricas PRECISAM do average para multiclasse
    precision = precision_score(y_true, y_pred, average=average)
    recall = recall_score(y_true, y_pred, average=average)
    f1 = f1_score(y_true, y_pred, average=average)

    return accuracy, precision, recall, f1
# Plota a matriz de confusão
def plotar_matriz_confusao(y_true, y_pred, title):

    # Matriz de confusão
    cm = confusion_matrix(y_true, y_pred)
    # Plot da matriz de confusão
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[0, 1,2,3],
                yticklabels=[0, 1,2,3])
    plt.xlabel('Predição')
    plt.ylabel('Valor Real')
    plt.title(title)
    plt.show()


In [ ]:
# Bagging - Combinação de 100 árvores independentes
bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                                  n_estimators=100,
                                  random_state=42)
bagging_model.fit(X_train_scaled, y_train)
y_pred_bagging = bagging_model.predict(X_test_scaled)
# Boosting
boosting_model = AdaBoostClassifier(n_estimators=100,
                                    random_state=42)
boosting_model.fit(X_train_scaled, y_train)
y_pred_boosting = boosting_model.predict(X_test_scaled)
# Definindo modelos base para ensembles para VOTING e STACKING
base_models = [
      ('dt', DecisionTreeClassifier(random_state=42)),
      ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
      ('knn', KNeighborsClassifier()),
      ('svm', SVC())
]
# Voting - Votação entre os modelos de base
voting_model = VotingClassifier(estimators=base_models,
                                voting='hard')
voting_model.fit(X_train_scaled, y_train)
y_pred_voting = voting_model.predict(X_test_scaled)
# Stacking
stacking_model = StackingClassifier(estimators=base_models,
                                    final_estimator=LogisticRegression())
stacking_model.fit(X_train_scaled, y_train)
y_pred_stacking = stacking_model.predict(X_test_scaled)
# XGBoost
xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)
# Avaliação dos modelos
accuracy_bagging, precision_bagging, recall_bagging, f1_bagging = avaliar_modelo(y_test, y_pred_bagging, average ='micro')
accuracy_boosting, precision_boosting, recall_boosting, f1_boosting = avaliar_modelo(y_test, y_pred_boosting, average ='micro')
accuracy_voting, precision_voting, recall_voting, f1_voting = avaliar_modelo(y_test, y_pred_voting, average ='micro')
accuracy_stacking, precision_stacking, recall_stacking, f1_stacking = avaliar_modelo(y_test, y_pred_stacking, average ='micro')
accuracy_xgb, precision_xgb, recall_xgb, f1_xgb = avaliar_modelo(y_test, y_pred_xgb,average ='micro' )
print(f"Bagging:	 \tAcurácia: {accuracy_bagging:.4f},         Precisão: {precision_bagging:.4f},         Recall: {recall_bagging:.4f},         F1-Score: {f1_bagging:.4f}")
print(f"Boosting:	 \tAcurácia: {accuracy_boosting:.4f},         Precisão: {precision_boosting:.4f},         Recall: {recall_boosting:.4f},         F1-Score: {f1_boosting:.4f}")
print(f"Voting:	   \tAcurácia: {accuracy_voting:.4f},         Precisão: {precision_voting:.4f},         Recall: {recall_voting:.4f},         F1-Score: {f1_voting:.4f}")
print(f"Stacking:	 \tAcurácia: {accuracy_stacking:.4f},         Precisão: {precision_stacking:.4f},         Recall: {recall_stacking:.4f},         F1-Score: {f1_stacking:.4f}")
print(f"XGBoost:	 \tAcurácia: {accuracy_xgb:.4f},         Precisão: {precision_xgb:.4f},         Recall: {recall_xgb:.4f},         F1-Score: {f1_xgb:.4f}")

Aqui conseguimos observar que entre os modelos Bagging, Boosting, Voting, Stacking e XGBoost o modelo que apresentou melhor resultado na classificação foi o Stacking com parametros acima de 0.93

In [ ]:
# Plota as matrizes de confusão para cada modelo

plotar_matriz_confusao(y_test, y_pred_bagging, "Matriz de Confusão - Bagging")
plotar_matriz_confusao(y_test, y_pred_boosting, "Matriz de Confusão - Boosting")
plotar_matriz_confusao(y_test, y_pred_voting, "Matriz de Confusão - Voting")
plotar_matriz_confusao(y_test, y_pred_stacking, "Matriz de Confusão - Stacking")
plotar_matriz_confusao(y_test, y_pred_xgb, "Matriz de Confusão - XGBoost")

In [ ]:
sns.pairplot(data=data, vars =('Precipitation (mm day-1)',	'Specific Humidity at 2 Meters (g/kg)',	'Temperature at 2 Meters (C)',	'Yield'),              hue='crop_encoded' );

In [ ]:

mapeamento = dict(zip(le.classes_, range(len(le.classes_))))
print(mapeamento)


Atraves do modelo de matris de confusão e da ferramenta do seaborn pairplot podemos observar que existem duas classes que são mais facilmente confundidas pelos modelos classes '0' e '3'

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier

##Perceptron e MLPerceptron

Agora utilizaremos o modelo de um ou mais neuronios para a classificação do modelo e avaliaremos seus resultados

In [ ]:
# Cria um perceptron e o treina (com o fit())
p = Perceptron( random_state=42 )
p.fit(X_train, y_train)
# Salva as predições do TESTE na variável y_pred
y_pred = p.predict(X_test)
# Matriz de confusão do perceptron
from sklearn.metrics import confusion_matrix
def matriz_confusao(teste_labels, teste_preds, labels):
  cm = confusion_matrix(teste_labels, teste_preds, labels=labels)
  disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                display_labels=labels)
  disp.plot(cmap="Blues")
matriz_confusao(y_test, y_pred, p.classes_)

In [ ]:
#Arquitetura
mlp = MLPClassifier(
    hidden_layer_sizes=(10,8,),
    solver='lbfgs',
    alpha=1e-2,
    verbose=True,
    max_iter=100,
    random_state=42
)
# Treinamento
mlp.fit(X_train_scaled, y_train)
y_pred = mlp.predict(X_test_scaled)
# Avaliação
matriz_confusao(y_test, y_pred, mlp.classes_)
print("ACC Teste: ", accuracy_score(y_test, y_pred))

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(20,10,),
    verbose=False,
    max_iter=80,
    random_state=42,
    activation="logistic",
    solver="lbfgs"
)
mlp.fit(X_train_scaled, y_train)
y_pred = mlp.predict(X_test_scaled)
print("ACC Teste: ", accuracy_score(y_test, y_pred))
matriz_confusao(y_test, y_pred, mlp.classes_)

É possivel observar que utilizando o MLPerceptron na ativação 'logistic' o resultado foi relativamente melhor que o sua ativação default e que o Perceptron.
Através de testes os parametros ajustados conseguiram uma acuracia de 0.718   

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_scaled)
shap.summary_plot(shap_values, X_test_scaled, feature_names=X_test.columns.tolist())

In [ ]:
import lime
import lime.lime_tabular

# 1. Criar o explicador
# training_data deve ser o seu X_train_scaled (em formato numpy)
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=X_test.columns.tolist(),
    class_names=mlp.classes_.tolist(), # ou xgb_model.classes_
    mode='classification' # ou 'regression'
)

# 2. Escolher um exemplo específico do teste para explicar (ex: o primeiro da lista)
i = 0

# 3. Gerar a explicação
# O LIME precisa de uma função que receba os dados e retorne probabilidades
exp = explainer.explain_instance(
    data_row=X_test_scaled[i],
    predict_fn=xgb_model.predict_proba # Use o seu modelo aqui
)

# 4. Mostrar o resultado
exp.show_in_notebook(show_table=True)

Atraves dos metodos Shap e Lime podemos ver a importancia das varaiveis no resultado final e observar que a classificação é paltada principalmente na variavel Yield

##PREDIÇÃO DE YIELD

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from pycaret.regression import *
train_two,test = train_test_split( data_predict.drop(['crop_encoded'], axis = 1), test_size=0.2, random_state=42)

reg = setup(
    data = train_two,
    target = 'Yield',     # O PyCaret já vai TIRAR o Yield das entradas aqui
    session_id = 42,
    preprocess = True
)

# 3. VERIFICAÇÃO DEFINITIVA
# Rode isso para ver se o Yield sumiu das entradas:
cols_entrada = get_config('X_train').columns
print("Variáveis que o modelo vai usar:", cols_entrada)

if 'Yield' in cols_entrada:
    print("ERRO: O Yield ainda está lá!")
else:
    print("SUCESSO: O Yield foi removido das entradas e é apenas o Target.")

Foi treinado varios modelos de predição para identificar o melhor

In [ ]:
compare_models(sort="R2")

Comparando o desempenho dos modelos treinados o modelo de Extra Trees Regressor apresentou melhores resultados de predição

In [ ]:
et = create_model("et")

Treinado o modelo de Extra Trees Regressor obtivemos bons resultados como mostrado na celula executada acima

In [ ]:
tuned_et = tune_model(et)

após a tentativa de melhorar o modelo atraves do tuned é possivel notar que o modelo não obteve melhora significativa no desempenho que o original

In [ ]:
plot_model(et, plot = "error")

In [ ]:
##comparar residuos
plot_model(et, plot = "residuals")

Conseguimos analisar o erro cometido pelo modelo e observamos que o r2 está proximo de 0.99

In [ ]:
predict_model(et, data=test)

Testando o modelo e fazendo predições obtemos 0.0.9941 de R2

In [ ]:
et_modelo_final = finalize_model(et)

Criamos o modelo final para produção

In [ ]:
save_model(et_modelo_final, "modelo_et_predict")

Salvamos em um arquivo .pkl para utilizar em outras ocaziões

#Modelo de Random Forest Regressor

In [ ]:
rf = create_model("rf")
plot_model(rf, plot = "error")
plot_model(rf, plot = "residuals")
predict_model(rf, data=test)

#Modelo de Lasso Least Angle Regression

In [ ]:
llar = create_model("llar")
plot_model(llar, plot = "error")
plot_model(llar, plot = "residuals")
predict_model(llar, data=test)

#Modelo de Least Angle Regression

In [ ]:
llar = create_model("llar")
plot_model(llar, plot = "error")
plot_model(llar, plot = "residuals")
predict_model(llar, data=test)

#Modelo de Gradient Boosting Regressor

In [ ]:
gbr = create_model("gbr")
plot_model(gbr, plot = "error")
plot_model(gbr, plot = "residuals")
predict_model(gbr, data=test)

##CONCLUSÃO

Através de técnicas de data Science constatamos que a base é solida sem dados faltantes ou duplicados , porem com poucos dados disponíveis.
A remoção de uma variável foi possível por sua alta correlação com outra não interferindo nos modelos de clusterização ou predição.
Aplicando técnicas de machine learn para realizar a clusterização conseguimos constatar que na fazenda em que os dados foram colhidos as variáveis não interferem tanto quando o tipo de plantio que foi cultivado, podemos observar que existem uma distribuição dos cultivos quase que uniforme mesmo com a variação dos parâmetros , ou seja o cultivo de Oil palm fruit tem uma produção consideravelmente maior por hectare que os demais cultivos.
O modelo de predição se comportou bem aos dados tendo uma boa performasse quando utilizado o modelo de Extra Trees Regressor , alcançando R2 proximo de 0.99 , foi o modelo indicado pela biblioteca Pycaret , porem foi treinado mais 4 modelos diferentes com ('rf', 'llar', 'lr', 'gbr') onde obtiveram resultados semelhantes

In [ ]:
from pycaret.regression import load_model, predict_model
import pandas as pd

# 1. Carregar o modelo
modelo = load_model('modelo_et_predict')

# 2. Novos dados
dados_novos = pd.DataFrame([{
    'Crop':'Cocoa, beans',
    'Specific Humidity at 2 Meters (g/kg)': 19.21,
    'Temperature at 2 Meters (C)': 22.05,
    'Precipitation (mm day-1)': 20000,


}])

# 3. Fazer a predição
previsoes = predict_model(modelo, data=dados_novos)

# O resultado aparecerá em uma nova coluna chamada 'prediction_label'
print(previsoes)


Teste de importação de modelo preditivo

In [ ]:
#Previsao de produção para o terrono de 20 hectares
predicao = previsoes['prediction_label'].iloc[0]
producao_total = predicao * 200
print(f"A produção total caso seja plantado em todos os 200 hectares é de {producao_total} toneladas")

##Modelo para predição de varios tipos de cultivo em varias condições de solo, de acordo com sua area plantada

In [ ]:
colunas_predicao = ['Cultivo','Area_plantada','Producao_estimada']

data_set = pd.DataFrame(columns = colunas_predicao)

escolha = input("Deseja adicionar algum dado? S/N").upper()

while escolha == 'S':
  modelo = load_model('modelo_et_predict')

  area_plantada = float(input("Qual area será destinada ao plantio (hectares)?"))
  cultivo = input("Qual cultivo ?")
  humidade = input("Qual a humidade do solo a 2 Metros (g/kg)")
  temperatura = input("Qual a temperatura a 2 Metros (c)")
  precipitacao = input("Qual a precipitação ? (mm day-1)")

  dados_predic = pd.DataFrame([{
    'Crop': cultivo,
    'Specific Humidity at 2 Meters (g/kg)': humidade,
    'Temperature at 2 Meters (C)': temperatura,
    'Precipitation (mm day-1)': precipitacao,
  }])

  previsoes = predict_model(modelo, data=dados_predic)

  preducao_estimada = (previsoes['prediction_label'].iloc[0])*area_plantada

  dados_n = pd.DataFrame({
      'Cultivo':[cultivo],
      'Area_plantada': [area_plantada],
      'Producao_estimada': [preducao_estimada]

  })
  data_set = pd.concat([dados_n,data_set], ignore_index= True)

  escolha = input("Deseja adicionar mais algum dado? S/N").upper()

print(data_set)




